# 03 Models: training, evaluation and interpretation

## 1. Training and tuning
Dummy baseline, logistic regression, random forest and LightGBM on feature sets A and B. Imbalance handling is compared (none vs class_weight vs SMOTENC inside an imblearn pipeline, training data only). Hyperparameters are tuned with RandomizedSearchCV using the validation period (2014). The 2015 test set is not used in this notebook.

#### Imports and load train + validation only

In [ ]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = str(os.cpu_count())   # avoids a joblib core-count warning on Windows
import sys, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import Markdown, display
sys.path.append("../src")

model_data = pd.read_parquet("../data/processed/model_data.parquet")
train = model_data[model_data["split"] == "train"].copy()
valid = model_data[model_data["split"] == "valid"].copy()
y_train = train["default"].values
y_valid = valid["default"].values

os.makedirs("../models", exist_ok=True)
print("train:", train.shape, "| valid:", valid.shape)
print("Train default rate:", round(y_train.mean() * 100, 2), "% | Validation default rate:", round(y_valid.mean() * 100, 2), "%")
print("The 2015 test split is intentionally NOT loaded in this notebook.")

#### Build the preprocessed matrices (preprocessing is fitted on train only, exactly as in the feature pipeline)

In [ ]:
from features import build_preprocessing_pipeline, get_X
from evaluation import evaluate_scores
from sklearn.metrics import roc_auc_score

data = {}
for fs in ["A", "B"]:
    prep_raw = build_preprocessing_pipeline(fs, scale=False)
    prep_scaled = build_preprocessing_pipeline(fs, scale=True)
    Xr_tr = prep_raw.fit_transform(get_X(train, fs))
    Xr_va = prep_raw.transform(get_X(valid, fs))
    Xs_tr = prep_scaled.fit_transform(get_X(train, fs))
    Xs_va = prep_scaled.transform(get_X(valid, fs))
    data[fs] = {"prep_raw": prep_raw, "prep_scaled": prep_scaled,
                "raw": (Xr_tr, Xr_va), "scaled": (Xs_tr, Xs_va)}
    print(f"Set {fs}: train {Xr_tr.shape}, valid {Xr_va.shape}")

#### Dummy baseline

In [ ]:
from sklearn.dummy import DummyClassifier

results = []
for fs in ["A", "B"]:
    X_tr, X_va = data[fs]["raw"]
    dummy = DummyClassifier(strategy="prior").fit(X_tr, y_train)
    p = dummy.predict_proba(X_va)[:, 1]
    m = evaluate_scores(y_valid, p)
    results.append({"model": "Dummy", "feature_set": fs, "imbalance": "-",
                    "train_roc_auc": 0.5, "valid_roc_auc": m["roc_auc"], "valid_pr_auc": m["pr_auc"],
                    "valid_gini": m["gini"], "valid_ks": m["ks"], "minutes": 0.0, "best_params": {}})

print("Dummy baseline on validation: ROC-AUC", round(m["roc_auc"], 3), "| PR-AUC", round(m["pr_auc"], 3),
      "| KS", round(m["ks"], 3))
print("Accuracy of always predicting 'no default' on validation:", round((1 - y_valid.mean()) * 100, 1), "%")
print("-> high accuracy, zero usefulness: this is why we do not use accuracy.")

### Model builder

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTENC

LGBM_BASE = dict(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=100,
                 subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                 random_state=42, n_jobs=-1, verbose=-1)

CAT_PREFIXES = ("missingindicator_", "home_ownership_", "verification_status_",
                "purpose_", "addr_state_", "application_type_")

def cat_indices(columns):
    """Positions of 0/1 and category columns (needed by SMOTENC)."""
    return [i for i, c in enumerate(columns) if c.startswith(CAT_PREFIXES)]

def build_estimator(kind, strategy, columns):
    """kind: 'lr', 'rf' or 'lgbm'. strategy: 'none', 'class_weight' or 'smote'."""
    if kind == "lr":
        return LogisticRegression(class_weight="balanced", max_iter=2000)
    if kind == "rf":
        clf = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42,
                                     class_weight="balanced_subsample" if strategy == "class_weight" else None)
    elif kind == "lgbm":
        clf = LGBMClassifier(**LGBM_BASE, class_weight="balanced" if strategy == "class_weight" else None)
    if strategy == "smote":
        return ImbPipeline([
            ("smote", SMOTENC(categorical_features=cat_indices(columns), sampling_strategy=0.5, random_state=42)),
            ("clf", clf),
        ])
    return clf

#### Imbalance experiment (set A, LightGBM, same settings for all three options)

In [ ]:
Xs_tr, Xs_va = data["A"]["scaled"]
print("Columns treated as categorical by SMOTENC:", len(cat_indices(Xs_tr.columns)), "of", Xs_tr.shape[1])

rows = []
for strategy in ["none", "class_weight", "smote"]:
    t0 = time.time()
    est = build_estimator("lgbm", strategy, list(Xs_tr.columns))
    est.fit(Xs_tr, y_train)
    p = est.predict_proba(Xs_va)[:, 1]
    m = evaluate_scores(y_valid, p)
    rows.append({"strategy": strategy, "roc_auc": m["roc_auc"], "pr_auc": m["pr_auc"], "ks": m["ks"],
                 "mean_predicted_prob": p.mean(), "actual_default_rate": y_valid.mean(),
                 "minutes": (time.time() - t0) / 60})
    print("done:", strategy)

imb = pd.DataFrame(rows).set_index("strategy")
print()
print(imb.round(4).to_string())

best_auc = imb["roc_auc"].max()
order = ["none", "class_weight", "smote"]
STRATEGY = [s for s in order if imb.loc[s, "roc_auc"] >= best_auc - 0.002][0]

display(Markdown(
    f"**Imbalance comparison (LightGBM, set A, validation 2014):** ROC-AUC is "
    f"{imb.loc['none','roc_auc']:.4f} with no handling, {imb.loc['class_weight','roc_auc']:.4f} with class_weight and "
    f"{imb.loc['smote','roc_auc']:.4f} with SMOTENC. The average predicted default probability is "
    f"{imb.loc['none','mean_predicted_prob']:.3f}, {imb.loc['class_weight','mean_predicted_prob']:.3f} and "
    f"{imb.loc['smote','mean_predicted_prob']:.3f} versus an actual rate of {y_valid.mean():.3f}, so resampling and "
    f"weighting distort probabilities and must be recalibrated before use. Rule applied: choose the simplest option within "
    f"0.002 AUC of the best, so **{STRATEGY}** is used for the random forest and LightGBM."))

**Note:** in this run only class_weight visibly distorted predicted probabilities (average 0.457 vs an actual rate of 0.137). No handling (0.131) and SMOTENC at a 0.5 ratio (0.142) stayed close to the actual rate. No handling was chosen by the rule; SMOTENC also took about 4 minutes and scored slightly lower.

#### The tuning helper 

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV, GridSearchCV
from sklearn.pipeline import Pipeline

def tune_and_finalize(model_name, fs, kind, param_space, n_iter=10):
    strategy = "class_weight" if kind == "lr" else STRATEGY
    key = "scaled" if (kind == "lr" or strategy == "smote") else "raw"
    prep = data[fs]["prep_scaled" if key == "scaled" else "prep_raw"]
    X_tr, X_va = data[fs][key]

    est = build_estimator(kind, strategy, list(X_tr.columns))
    prefix = "clf__" if strategy == "smote" else ""
    space = {prefix + k: v for k, v in param_space.items()}

    # one split: rows of the train period are always "train", rows of the validation period are the "test fold"
    X_all = pd.concat([X_tr, X_va])
    y_all = np.concatenate([y_train, y_valid])
    cv = PredefinedSplit(np.r_[np.full(len(X_tr), -1), np.zeros(len(X_va))].astype(int))

    if kind == "lr":
        search = GridSearchCV(est, space, scoring="roc_auc", cv=cv, refit=False)
    else:
        search = RandomizedSearchCV(est, space, n_iter=n_iter, scoring="roc_auc", cv=cv,
                                    refit=False, random_state=42)
    t0 = time.time()
    search.fit(X_all, y_all)
    best = search.best_params_

    # final model: best settings, trained on the training period only
    final = clone(est).set_params(**best).fit(X_tr, y_train)
    clf = final.named_steps["clf"] if strategy == "smote" else final
    p_tr = final.predict_proba(X_tr)[:, 1]
    p_va = final.predict_proba(X_va)[:, 1]
    mv = evaluate_scores(y_valid, p_va)

    full = Pipeline([("prep", prep), ("model", clf)])
    joblib.dump(full, f"../models/{fs}_{model_name}.joblib")

    top = pd.DataFrame(search.cv_results_)[["params", "mean_test_score"]].sort_values("mean_test_score", ascending=False)
    print(f"--- {model_name} | set {fs} | imbalance: {strategy} ---")
    print("Top candidates (validation ROC-AUC):")
    print(top.head(3).to_string(index=False))
    print(f"Best settings: {best}")
    print(f"Train ROC-AUC {roc_auc_score(y_train, p_tr):.4f} | Validation ROC-AUC {mv['roc_auc']:.4f} | "
          f"PR-AUC {mv['pr_auc']:.4f} | KS {mv['ks']:.4f} | {(time.time() - t0) / 60:.1f} min\n")

    results.append({"model": model_name, "feature_set": fs, "imbalance": strategy,
                    "train_roc_auc": roc_auc_score(y_train, p_tr), "valid_roc_auc": mv["roc_auc"],
                    "valid_pr_auc": mv["pr_auc"], "valid_gini": mv["gini"], "valid_ks": mv["ks"],
                    "minutes": (time.time() - t0) / 60,
                    "best_params": {k.replace("clf__", ""): v for k, v in best.items()}})

#### Logistic regression (quick; 5 values of C, which controls regularization strength; smaller means stronger)

In [ ]:
for fs in ["A", "B"]:
    tune_and_finalize("LogisticRegression", fs, "lr", {"C": [0.001, 0.01, 0.1, 1, 10]})

#### Random forest

In [ ]:
rf_space = {"max_depth": [6, 8, 10, 12],
            "min_samples_leaf": [50, 100, 200, 400],
            "max_features": ["sqrt", 0.3, 0.5]}
for fs in ["A", "B"]:
    tune_and_finalize("RandomForest", fs, "rf", rf_space, n_iter=8)

#### LightGBM

In [ ]:
lgbm_space = {"n_estimators": [200, 400, 600],
              "learning_rate": [0.02, 0.05, 0.1],
              "num_leaves": [15, 31, 63],
              "min_child_samples": [50, 100, 200],
              "subsample": [0.7, 0.9, 1.0],
              "colsample_bytree": [0.6, 0.8, 1.0],
              "reg_lambda": [0, 1, 10]}
for fs in ["A", "B"]:
    tune_and_finalize("LightGBM", fs, "lgbm", lgbm_space, n_iter=12)

#### Results table, chart and saved files

In [ ]:
res = pd.DataFrame(results)
show = res.drop(columns=["best_params"]).copy()
for c in show.columns:
    if show[c].dtype == float:
        show[c] = show[c].round(4)
print(show.to_string(index=False))
show.to_csv("../reports/model_results_validation.csv", index=False)

def to_plain(o):
    return o.item() if hasattr(o, "item") else str(o)

with open("../reports/best_params.json", "w") as f:
    json.dump({f"{r['feature_set']}_{r['model']}": r["best_params"] for r in results if r["model"] != "Dummy"},
              f, indent=2, default=to_plain)
with open("../reports/model_config.json", "w") as f:
    json.dump({"imbalance_strategy_for_trees": STRATEGY, "logistic_regression_imbalance": "class_weight",
               "tuning_split": "validation (2014)", "final_models_trained_on": "train (2012-2013)"}, f, indent=2)

order_models = ["Dummy", "LogisticRegression", "RandomForest", "LightGBM"]
pivot = res.pivot(index="model", columns="feature_set", values="valid_roc_auc").reindex(order_models)
ax = pivot.plot(kind="bar", figsize=(8, 4.5))
for c in ax.containers:
    ax.bar_label(c, fmt="%.3f", fontsize=8)
ax.set_ylabel("Validation ROC-AUC (2014)")
ax.set_title("Validation ROC-AUC by model and feature set")
ax.legend(title="Feature set")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("../reports/figures/E1_validation_auc_by_model.png", dpi=150)
plt.show()

real = res[res["model"] != "Dummy"]
bA = real[real["feature_set"] == "A"].sort_values("valid_roc_auc", ascending=False).iloc[0]
bB = real[real["feature_set"] == "B"].sort_values("valid_roc_auc", ascending=False).iloc[0]
display(Markdown(
    f"**Takeaway (validation 2014):** the best application-only model (set A) is {bA['model']} with ROC-AUC "
    f"{bA['valid_roc_auc']:.3f} (KS {bA['valid_ks']:.3f}); adding LendingClub's own grade, rate and installment (set B) "
    f"lifts the best model, {bB['model']}, to {bB['valid_roc_auc']:.3f} (KS {bB['valid_ks']:.3f}). "
    f"The dummy baseline scores 0.500."))

### Training summary
Four model types were trained on feature sets A and B: a dummy baseline, L2 logistic regression (balanced class weights, scaled inputs), a depth-limited random forest and LightGBM. Imbalance handling (none, class_weight, SMOTENC applied inside an imblearn pipeline on training data only) was compared on validation. All three gave nearly identical ranking performance (ROC-AUC 0.661 to 0.663), so the simplest option, no handling, was kept for the random forest and LightGBM. Class weighting visibly inflated predicted probabilities (0.457 vs an actual 0.137), so any model using weights must be recalibrated before its scores are used as default probabilities (notebook 04). Hyperparameters were tuned with RandomizedSearchCV using the 2014 validation period only. Final models were trained on 2012-2013 loans; the 2015 test set was not used in this notebook.

Validation ROC-AUC: LightGBM 0.664 (set A) and 0.680 (set B), random forest 0.657 and 0.678, logistic regression 0.655 and 0.675. Results are in `reports/model_results_validation.csv` and chart `E1_validation_auc_by_model.png`. Fitted pipelines (preprocessing + model) are saved in `models/`.

## 2. Evaluation on the 2015 test set
Models are frozen. Test loans are scored once and the predictions saved; every later cell works from the saved predictions. Thresholds are chosen on the validation set (2014) and applied unchanged to the test set.

#### Imports, data and models

In [ ]:
import os, sys, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import Markdown, display
from sklearn.metrics import (roc_auc_score, precision_recall_curve, confusion_matrix, brier_score_loss)
from sklearn.calibration import calibration_curve

sys.path.append("../src")
import features, feature_sets, evaluation
from features import get_X
from evaluation import evaluate_scores

sns.set_theme(style="whitegrid")

model_data = pd.read_parquet("../data/processed/model_data.parquet")
train_rate = model_data.loc[model_data["split"] == "train", "default"].mean()
valid = model_data[model_data["split"] == "valid"].copy()
test = model_data[model_data["split"] == "test"].copy()
print("valid:", valid.shape, "| test:", test.shape)

MODEL_NAMES = ["LogisticRegression", "RandomForest", "LightGBM"]
pipelines = {}
for fs in ["A", "B"]:
    for m in MODEL_NAMES:
        pipelines[(fs, m)] = joblib.load(f"../models/{fs}_{m}.joblib")
print("loaded", len(pipelines), "fitted pipelines")

#### Score validation and test once, and save the predictions

In [ ]:
t0 = time.time()
pred = {}
for split_name, df in [("valid", valid), ("test", test)]:
    out = df[["id", "issue_d", "default", "grade", "int_rate", "loan_amnt"]].copy()
    for (fs, m), pipe in pipelines.items():
        out[f"{fs}_{m}"] = pipe.predict_proba(get_X(df, fs))[:, 1]
    pred[split_name] = out
    out.to_parquet(f"../data/processed/{split_name}_predictions.parquet", index=False)
    print("scored", split_name, out.shape)

pv, pt = pred["valid"], pred["test"]
print(f"Done in {(time.time() - t0) / 60:.1f} min. Predictions saved in data/processed/.")
print("Test default rate:", round(pt["default"].mean() * 100, 2), "%")

#### Headline metrics for every model (validation vs test)

In [ ]:
GRADE_NUM = {"A": 1, "B": 2, "C": 3, "D": 4, "E": 5, "F": 6, "G": 7}

def lc_scores(df):
    return {"LendingClub grade alone": df["grade"].map(GRADE_NUM).astype(float).values,
            "LendingClub interest rate alone": df["int_rate"].values}

def metrics_row(name, fs, sv, st):
    mv = evaluate_scores(pv["default"], sv)
    mt = evaluate_scores(pt["default"], st)
    return {"model": name, "feature_set": fs, "valid_roc_auc": mv["roc_auc"], "test_roc_auc": mt["roc_auc"],
            "auc_change": mt["roc_auc"] - mv["roc_auc"], "test_pr_auc": mt["pr_auc"],
            "test_gini": mt["gini"], "test_ks": mt["ks"]}

rows = []
for fs in ["A", "B"]:
    for m in MODEL_NAMES:
        col = f"{fs}_{m}"
        rows.append(metrics_row(m, fs, pv[col], pt[col]))
lcv, lct = lc_scores(pv), lc_scores(pt)
for name in lcv:
    rows.append(metrics_row(name, "-", lcv[name], lct[name]))
rows.append({"model": "Dummy", "feature_set": "-", "valid_roc_auc": 0.5, "test_roc_auc": 0.5, "auc_change": 0.0,
             "test_pr_auc": pt["default"].mean(), "test_gini": 0.0, "test_ks": 0.0})

res = pd.DataFrame(rows)
print(res.round(4).to_string(index=False))
res.round(4).to_csv("../reports/model_results_test.csv", index=False)
print("\nNo-skill PR-AUC on test equals the default rate:", round(pt["default"].mean(), 4))

# chart
plot_df = res[res["feature_set"].isin(["A", "B"])]
pivot = plot_df.pivot(index="model", columns="feature_set", values="test_roc_auc").reindex(MODEL_NAMES)
ax = pivot.plot(kind="bar", figsize=(8, 4.5))
for c in ax.containers:
    ax.bar_label(c, fmt="%.3f", fontsize=8)
ref_rate = res.loc[res["model"] == "LendingClub interest rate alone", "test_roc_auc"].iloc[0]
ref_grade = res.loc[res["model"] == "LendingClub grade alone", "test_roc_auc"].iloc[0]
ax.axhline(ref_rate, color="red", linestyle="--", linewidth=1, label=f"LC interest rate alone ({ref_rate:.3f})")
ax.axhline(ref_grade, color="gray", linestyle=":", linewidth=1, label=f"LC grade alone ({ref_grade:.3f})")
ax.set_ylim(0.5, max(pivot.max().max(), ref_rate, ref_grade) + 0.03)
ax.set_ylabel("Test ROC-AUC (2015)")
ax.set_title("Test ROC-AUC by model and feature set (axis starts at 0.5)")
ax.legend(title="Feature set / reference", fontsize=8)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("../reports/figures/F1_test_auc_by_model.png", dpi=150)
plt.show()

chg = plot_df["auc_change"]
display(Markdown(f"**Takeaway (drift):** from validation (2014) to test (2015) the ROC-AUC of the six models changed by "
                 f"{chg.min():+.3f} to {chg.max():+.3f} (average {chg.mean():+.3f}), with the default rate rising from "
                 f"{pv['default'].mean() * 100:.1f}% to {pt['default'].mean() * 100:.1f}%."))

#### Pick the best model per feature set (by validation only) and measure uncertainty

In [ ]:
val_res = pd.read_csv("../reports/model_results_validation.csv")
val_res = val_res[val_res["model"] != "Dummy"]
chosen = {fs: val_res[val_res["feature_set"] == fs].sort_values("valid_roc_auc", ascending=False).iloc[0]["model"]
          for fs in ["A", "B"]}
print("Chosen by validation ROC-AUC:", chosen)

colA, colB = f"A_{chosen['A']}", f"B_{chosen['B']}"


def bootstrap_auc(y, scores, n=500, seed=42):
    """Paired bootstrap: every score is evaluated on the same resampled loans, so differences are valid."""
    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    scores = {k: np.asarray(v) for k, v in scores.items()}
    draws = {k: [] for k in scores}
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        for k, s in scores.items():
            draws[k].append(roc_auc_score(y[idx], s[idx]))
    return {k: np.array(v) for k, v in draws.items()}


y_test = pt["default"].values
scores = {"A": pt[colA].values, "B": pt[colB].values, "LC interest rate": pt["int_rate"].values}
draws = bootstrap_auc(y_test, scores, n=500)
aucA, aucB = roc_auc_score(y_test, scores["A"]), roc_auc_score(y_test, scores["B"])
aucR = roc_auc_score(y_test, scores["LC interest rate"])

def ci(v): return np.percentile(v, [2.5, 97.5])
ciA, ciB, ciR = ci(draws["A"]), ci(draws["B"]), ci(draws["LC interest rate"])
ciD = ci(draws["B"] - draws["A"])                      # set B minus set A
ciBR = ci(draws["B"] - draws["LC interest rate"])      # set B minus LendingClub interest rate alone
ciAR = ci(draws["A"] - draws["LC interest rate"])      # set A minus LendingClub interest rate alone

ci_table = pd.DataFrame([
    ("Set A ROC-AUC", aucA, *ciA), ("Set B ROC-AUC", aucB, *ciB), ("LendingClub interest rate ROC-AUC", aucR, *ciR),
    ("Set B minus Set A", aucB - aucA, *ciD), ("Set B minus LendingClub interest rate", aucB - aucR, *ciBR),
    ("Set A minus LendingClub interest rate", aucA - aucR, *ciAR),
], columns=["quantity", "estimate", "ci_95_low", "ci_95_high"])
print(ci_table.round(4).to_string(index=False))
ci_table.round(4).to_csv("../reports/test_auc_confidence_intervals.csv", index=False)

#### Precision, recall and confusion matrices at thresholds chosen on validation

In [ ]:
def thresholds_from_validation(y, p):
    prec, rec, th = precision_recall_curve(y, p)
    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
    return {"F1-optimal": th[np.argmax(f1)],
            "Flag riskiest 20%": np.quantile(p, 0.80),
            "Flag riskiest 10%": np.quantile(p, 0.90)}

rows = []
for fs in ["A", "B"]:
    col = f"{fs}_{chosen[fs]}"
    for rule, t in thresholds_from_validation(pv["default"], pv[col]).items():
        flag = pt[col] >= t
        tn, fp, fn, tp = confusion_matrix(pt["default"], flag).ravel()
        rows.append({"feature_set": fs, "model": chosen[fs], "threshold_rule": rule, "threshold": t,
                     "flagged_share_%": flag.mean() * 100, "precision": tp / (tp + fp), "recall": tp / (tp + fn),
                     "TN": tn, "FP": fp, "FN": fn, "TP": tp})
thr = pd.DataFrame(rows)
print(thr.round(3).to_string(index=False))
thr.round(4).to_csv("../reports/threshold_metrics_test.csv", index=False)
print("\nFor reference, the test default rate (precision of flagging everyone):", round(pt["default"].mean(), 4))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, fs in zip(axes, ["A", "B"]):
    r = thr[(thr["feature_set"] == fs) & (thr["threshold_rule"] == "Flag riskiest 20%")].iloc[0]
    cm = np.array([[r["TN"], r["FP"]], [r["FN"], r["TP"]]], dtype=int)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Not flagged", "Flagged"], yticklabels=["Repaid", "Defaulted"])
    ax.set_title(f"Set {fs}: {chosen[fs]}\nthreshold = validation top-20% cutoff ({r['threshold']:.3f})")
plt.tight_layout()
plt.savefig("../reports/figures/F2_confusion_matrices.png", dpi=150)
plt.show()

r = thr[(thr["feature_set"] == "B") & (thr["threshold_rule"] == "Flag riskiest 20%")].iloc[0]
display(Markdown(f"**Takeaway:** with the cutoff that flags the riskiest 20% of validation loans, set B ({chosen['B']}) flags "
                 f"{r['flagged_share_%']:.1f}% of 2015 loans, catching {r['recall'] * 100:.1f}% of defaults with "
                 f"{r['precision'] * 100:.1f}% precision (the 2015 default rate is {pt['default'].mean() * 100:.1f}%). "
                 f"The flagged share can differ from 20% because 2015 loans were riskier than 2014 loans."))

#### Calibration curves and Brier score

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

for fs in ["A", "B"]:
    col = f"{fs}_{chosen[fs]}"
    frac_pos, mean_pred = calibration_curve(pt["default"], pt[col], n_bins=10, strategy="quantile")
    axes[0].plot(mean_pred, frac_pos, marker="o", label=f"Set {fs}: {chosen[fs]}")
col = "A_LogisticRegression"
frac_pos, mean_pred = calibration_curve(pt["default"], pt[col], n_bins=10, strategy="quantile")
axes[1].plot(mean_pred, frac_pos, marker="o", color="tab:red", label="Set A: LogisticRegression (balanced weights)")

for ax, title in zip(axes, ["Chosen models (no class weighting)", "Logistic regression with balanced class weights"]):
    top = max(ax.get_xlim()[1], ax.get_ylim()[1])
    ax.plot([0, top], [0, top], "k--", label="Perfect calibration")
    ax.set_xlabel("Average predicted probability")
    ax.set_ylabel("Actual default rate")
    ax.set_title(title)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("../reports/figures/F3_calibration.png", dpi=150)
plt.show()

brier_const = brier_score_loss(pt["default"], np.full(len(pt), train_rate))
rows = []
for fs in ["A", "B"]:
    for m in MODEL_NAMES:
        col = f"{fs}_{m}"
        b = brier_score_loss(pt["default"], pt[col])
        rows.append({"model": m, "feature_set": fs, "brier": b, "skill_vs_constant_%": (1 - b / brier_const) * 100,
                     "mean_predicted_prob": pt[col].mean(), "actual_default_rate": pt["default"].mean()})
brier = pd.DataFrame(rows)
print("Brier score of a constant guess (training default rate):", round(brier_const, 5))
print(brier.round(4).to_string(index=False))
brier.round(5).to_csv("../reports/brier_test.csv", index=False)

lrA = brier[(brier["model"] == "LogisticRegression") & (brier["feature_set"] == "A")].iloc[0]
chA = brier[(brier["model"] == chosen["A"]) & (brier["feature_set"] == "A")].iloc[0]
display(Markdown(f"**Takeaway:** the chosen unweighted model (set A, {chosen['A']}) predicts an average default probability of "
                 f"{chA['mean_predicted_prob']:.3f} against an actual {chA['actual_default_rate']:.3f} on the test set, while "
                 f"class-weighted logistic regression predicts {lrA['mean_predicted_prob']:.3f}, so its scores rank loans well "
                 f"but are not true probabilities until recalibrated (notebook 04)."))

#### Lift by decile

In [ ]:
def lift_table(y, p, n_bins=10):
    d = pd.DataFrame({"y": np.asarray(y), "p": np.asarray(p)}).sort_values("p", ascending=False).reset_index(drop=True)
    d["decile"] = pd.qcut(np.arange(len(d)), n_bins, labels=False) + 1
    g = d.groupby("decile")["y"].agg(loans="count", defaults="sum")
    g["default_rate_%"] = g["defaults"] / g["loans"] * 100
    g["lift"] = g["default_rate_%"] / (d["y"].mean() * 100)
    g["cum_defaults_captured_%"] = g["defaults"].cumsum() / d["y"].sum() * 100
    return g

lifts = {}
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, fs in zip(axes, ["A", "B"]):
    col = f"{fs}_{chosen[fs]}"
    lifts[fs] = lift_table(pt["default"], pt[col])
    print(f"\nSet {fs}: {chosen[fs]} (decile 1 = riskiest 10% of loans)")
    print(lifts[fs].round(2).to_string())
    lifts[fs].round(3).to_csv(f"../reports/lift_by_decile_{fs}.csv")
    ax.bar(lifts[fs].index.astype(str), lifts[fs]["default_rate_%"])
    ax.axhline(pt["default"].mean() * 100, color="red", linestyle="--", label="Overall test default rate")
    ax.set_xlabel("Risk decile (1 = riskiest)")
    ax.set_title(f"Set {fs}: {chosen[fs]}")
    ax.legend(fontsize=8)
axes[0].set_ylabel("Actual default rate (%)")
plt.tight_layout()
plt.savefig("../reports/figures/F4_lift_by_decile.png", dpi=150)
plt.show()

lb = lifts["B"]
display(Markdown(f"**Takeaway:** for set B ({chosen['B']}) the riskiest decile of 2015 loans defaulted at "
                 f"{lb.loc[1, 'default_rate_%']:.1f}% (lift {lb.loc[1, 'lift']:.2f}) while the safest decile defaulted at "
                 f"{lb.loc[10, 'default_rate_%']:.1f}%; the riskiest 30% of loans contain {lb.loc[3, 'cum_defaults_captured_%']:.1f}% of all defaults."))

#### Answers to the business sub-questions

In [ ]:
rate_auc = res.loc[res["model"] == "LendingClub interest rate alone", "test_roc_auc"].iloc[0]
grade_auc = res.loc[res["model"] == "LendingClub grade alone", "test_roc_auc"].iloc[0]

def verdict(lo, hi):
    return "the interval excludes zero" if (lo > 0 or hi < 0) else "the interval includes zero, so the gap may be noise"

display(Markdown(
    f"**Can we rank risk using only application-time information?** Yes, modestly: set A ({chosen['A']}) reaches test ROC-AUC "
    f"{aucA:.3f} (95% interval {ciA[0]:.3f} to {ciA[1]:.3f}), KS {evaluate_scores(pt['default'], pt[colA])['ks']:.3f}.\n\n"
    f"**Do we add information beyond LendingClub's grade and interest rate?** The interest rate alone scores {rate_auc:.3f} "
    f"and the grade alone {grade_auc:.3f}. Set A scores {aucA:.3f} without using either "
    f"(A minus interest rate {aucA - aucR:+.3f}, 95% interval {np.percentile(draws['A'] - draws['LC interest rate'], 2.5):+.3f} to "
    f"{np.percentile(draws['A'] - draws['LC interest rate'], 97.5):+.3f}). Set B ({chosen['B']}) scores {aucB:.3f}: "
    f"B minus interest rate {aucB - aucR:+.3f} (95% interval {ciBR[0]:+.3f} to {ciBR[1]:+.3f}; {verdict(*ciBR)}), "
    f"B minus A {aucB - aucA:+.3f} (95% interval {ciD[0]:+.3f} to {ciD[1]:+.3f}; {verdict(*ciD)})."))

### Evaluation summary
All six models (logistic regression, random forest and LightGBM on feature sets A and B) were scored once on the 2015 test loans; predictions were saved and all further analysis used the saved predictions. Thresholds were chosen on the 2014 validation set and applied unchanged to the test set. Reported: ROC-AUC, PR-AUC, Gini, KS (`reports/model_results_test.csv`), validation vs test AUC for drift, precision/recall and confusion matrices at three thresholds (`threshold_metrics_test.csv`), calibration curves and Brier scores (`brier_test.csv`), lift by decile (`lift_by_decile_A.csv`, `lift_by_decile_B.csv`), bootstrap confidence intervals for AUC and for the B-minus-A difference, and comparison against LendingClub's own grade and interest rate used as scores.

Results are reported as obtained, without adjustment. Application-only models give modest discrimination, consistent with the expected high-0.6s ROC-AUC range. Limitations: only approved loans are observed (selection bias), one vintage window, no macro-economic factors, and joint applications appear only in the 2015 test period.

## 3. Interpretation
Logistic regression odds ratios, permutation importance, SHAP and partial dependence, all computed on the 2014 validation loans. Signs are checked against business expectations.

#### Load models and validation data

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from IPython.display import Markdown, display
from sklearn.inspection import permutation_importance
sys.path.append("../src")
import features, feature_sets
from features import get_X
sns.set_theme(style="whitegrid")

model_data = pd.read_parquet("../data/processed/model_data.parquet")
valid = model_data[model_data["split"] == "valid"].copy()
y_valid = valid["default"].values

G = {}
for fs in ["A", "B"]:
    gbm_pipe = joblib.load(f"../models/{fs}_LightGBM.joblib")
    lr_pipe = joblib.load(f"../models/{fs}_LogisticRegression.joblib")
    G[fs] = {"gbm": gbm_pipe.named_steps["model"], "lr": lr_pipe.named_steps["model"],
             "Xg": gbm_pipe.named_steps["prep"].transform(get_X(valid, fs)),
             "Xl": lr_pipe.named_steps["prep"].transform(get_X(valid, fs))}
    print("set", fs, "| LightGBM matrix:", G[fs]["Xg"].shape, "| logistic matrix:", G[fs]["Xl"].shape)

summary = {"perm_top5": {}, "shap_top5": {}, "lr_sign_matches": {}, "pdp_direction": {}}

#### Logistic regression odds ratios and the sign check

In [ ]:
EXPECTED = {"dti": 1, "revol_util": 1, "inq_last_6mths": 1, "delinq_2yrs": 1, "pub_rec": 1,
            "pub_rec_bankruptcies": 1, "fico_mid": -1, "log_annual_inc": -1, "loan_to_income": 1,
            "loan_amnt": 1, "int_rate": 1, "grade_num": 1, "installment_to_income": 1,
            "mort_acc": -1, "credit_history_years": -1}

def odds_table(fs):
    lr, cols = G[fs]["lr"], G[fs]["Xl"].columns
    t = pd.DataFrame({"feature": cols, "coef": lr.coef_[0]})
    t["odds_ratio"] = np.exp(t["coef"])
    t["abs_coef"] = t["coef"].abs()
    return t.sort_values("abs_coef", ascending=False).drop(columns="abs_coef").reset_index(drop=True)

def sign_check(t):
    rows = []
    for f, s in EXPECTED.items():
        if f in set(t["feature"]):
            c = float(t.loc[t["feature"] == f, "coef"].iloc[0])
            rows.append({"feature": f, "expected": "+" if s > 0 else "-", "coefficient": round(c, 3),
                         "odds_ratio": round(float(np.exp(c)), 3), "matches": (c > 0) == (s > 0)})
    return pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, fs in zip(axes, ["A", "B"]):
    t = odds_table(fs)
    t.round(4).to_csv(f"../reports/lr_odds_ratios_{fs}.csv", index=False)
    top = t.head(15).iloc[::-1]
    ax.barh(top["feature"], top["coef"], color=np.where(top["coef"] > 0, "tab:red", "tab:blue"))
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"Set {fs}: logistic regression coefficients (top 15)")
    ax.set_xlabel("Coefficient (red = raises default risk, blue = lowers)")
    sc = sign_check(t)
    summary["lr_sign_matches"][fs] = [int(sc["matches"].sum()), int(len(sc))]
    print(f"\n===== Set {fs}: strongest 12 effects (odds ratio per +1 standard deviation for numeric columns, per 0 to 1 for categories) =====")
    print(t.head(12).round(3).to_string(index=False))
    print(f"\nSign check against business expectations (set {fs}):")
    print(sc.to_string(index=False))
    bad = sc.loc[~sc["matches"], "feature"].tolist()
    display(Markdown(f"**Sign check, set {fs}:** {int(sc['matches'].sum())} of {len(sc)} coefficients have the expected sign"
                     + (f"; the exceptions ({', '.join(bad)}) can arise because correlated features (for example loan amount, loan-to-income and income) share their effect." if bad else ".")))
plt.tight_layout()
plt.savefig("../reports/figures/G1_lr_odds_ratios.png", dpi=150)
plt.show()

**Reading the coefficients:** the strongest effects match business logic. A higher FICO score is the biggest protective factor (odds ratio 0.70 per standard deviation in set A), while more recent credit inquiries (1.20), higher debt-to-income (1.13) and a higher loan-to-income ratio (1.12) raise default risk. Higher income lowers risk (0.84), credit-card loans are safer (0.74) and small-business loans riskier (1.33). In set B the interest rate is the strongest single driver (1.32). The coefficients with an unexpected sign (public records, bankruptcies, credit history length, and in set B utilization and loan amount) are all close to zero (absolute value at most 0.04), meaning they add almost no independent information once the other features are included; they do not contradict the business logic.

#### Permutation importance

In [ ]:
perm_tables = {}
for fs in ["A", "B"]:
    X = G[fs]["Xg"]
    idx = np.random.default_rng(42).choice(len(X), size=min(50000, len(X)), replace=False)
    r = permutation_importance(G[fs]["gbm"], X.iloc[idx], y_valid[idx], scoring="roc_auc",
                               n_repeats=5, random_state=42, n_jobs=1)
    perm_tables[fs] = pd.DataFrame({"feature": X.columns, "auc_drop": r.importances_mean,
                                    "std": r.importances_std}).sort_values("auc_drop", ascending=False).reset_index(drop=True)
    perm_tables[fs].round(5).to_csv(f"../reports/permutation_importance_{fs}.csv", index=False)
    summary["perm_top5"][fs] = perm_tables[fs].head(5)["feature"].tolist()
    print(f"Set {fs} done")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, fs in zip(axes, ["A", "B"]):
    top = perm_tables[fs].head(15).iloc[::-1]
    ax.barh(top["feature"], top["auc_drop"], xerr=top["std"])
    ax.set_title(f"Set {fs}: permutation importance (LightGBM, validation)")
    ax.set_xlabel("Drop in ROC-AUC when the feature is shuffled")
plt.tight_layout()
plt.savefig("../reports/figures/G2_permutation_importance.png", dpi=150)
plt.show()
for fs in ["A", "B"]:
    t = perm_tables[fs]
    display(Markdown(f"**Takeaway, set {fs}:** shuffling {t.loc[0, 'feature']} hurts ROC-AUC most (drop {t.loc[0, 'auc_drop']:.4f}), "
                     f"followed by {t.loc[1, 'feature']} ({t.loc[1, 'auc_drop']:.4f}) and {t.loc[2, 'feature']} ({t.loc[2, 'auc_drop']:.4f})."))

#### SHAP summary and dependence plots

In [ ]:
import shap
dep_features = {"A": ["dti", "fico_mid", "revol_util"], "B": ["int_rate", "dti", "fico_mid"]}
shap_top = {}
for fs in ["A", "B"]:
    Xs = G[fs]["Xg"].sample(n=5000, random_state=42)
    contrib = np.asarray(G[fs]["gbm"].booster_.predict(Xs, pred_contrib=True))
    sv = contrib[:, :-1]          # SHAP values in log-odds; the last column is the baseline
    shap_top[fs] = pd.Series(np.abs(sv).mean(axis=0), index=Xs.columns).sort_values(ascending=False)
    shap_top[fs].round(5).to_csv(f"../reports/shap_mean_abs_{fs}.csv", header=["mean_abs_shap"])
    summary["shap_top5"][fs] = shap_top[fs].head(5).index.tolist()

    shap.summary_plot(sv, Xs, max_display=15, show=False)
    plt.title(f"Set {fs}: SHAP summary (LightGBM, 5,000 validation loans)")
    plt.tight_layout()
    plt.savefig(f"../reports/figures/G3_shap_summary_{fs}.png", dpi=150, bbox_inches="tight")
    plt.show()

    for f in dep_features[fs]:
        shap.dependence_plot(f, sv, Xs, interaction_index=None, show=False)
        plt.title(f"Set {fs}: SHAP dependence for {f}")
        plt.tight_layout()
        plt.savefig(f"../reports/figures/G4_shap_dependence_{fs}_{f}.png", dpi=150, bbox_inches="tight")
        plt.show()
    display(Markdown(f"**Takeaway, set {fs}:** by mean absolute SHAP value the most influential features are "
                     f"{', '.join(shap_top[fs].head(5).index)}."))

#### Partial dependence for interest rate, DTI and FICO

In [ ]:
def manual_pdp(model, X, feature, n_points=25, sample=5000):
    Xs = X.sample(n=min(sample, len(X)), random_state=42).copy()
    grid = np.unique(np.quantile(X[feature].values, np.linspace(0.02, 0.98, n_points)))
    avg = []
    for v in grid:
        Xs[feature] = v
        avg.append(model.predict_proba(Xs)[:, 1].mean())
    return grid, np.array(avg)

pdp_features = ["int_rate", "dti", "fico_mid"]
expected_dir = {"int_rate": "rises", "dti": "rises", "fico_mid": "falls"}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, f in zip(axes, pdp_features):
    for fs, color in [("A", "tab:blue"), ("B", "tab:red")]:
        if f in G[fs]["Xg"].columns:
            grid, avg = manual_pdp(G[fs]["gbm"], G[fs]["Xg"], f)
            ax.plot(grid, avg * 100, color=color, label=f"Set {fs}")
            rho = pd.Series(avg).corr(pd.Series(grid), method="spearman")
            direction = "rises" if rho > 0.7 else ("falls" if rho < -0.7 else "is not monotonic")
            summary["pdp_direction"].setdefault(fs, {})[f] = direction
    ax.set_xlabel(f)
    ax.set_ylabel("Average predicted default probability (%)")
    ax.set_title(f"Partial dependence: {f}")
    ax.legend()
plt.tight_layout()
plt.savefig("../reports/figures/G5_partial_dependence.png", dpi=150)
plt.show()

lines = []
for f in pdp_features:
    for fs in ["A", "B"]:
        d = summary["pdp_direction"].get(fs, {}).get(f)
        if d:
            ok = "as expected" if d == expected_dir[f] else "NOT as expected"
            lines.append(f"{f} (set {fs}): predicted risk {d}, {ok}")
display(Markdown("**Partial-dependence sign check:** " + "; ".join(lines) + "."))

#### Save the interpretation summary

In [ ]:
with open("../reports/interpretation_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("Saved reports/interpretation_summary.json")

### Interpretation summary
Logistic regression odds ratios, LightGBM permutation importance, TreeSHAP summary and dependence plots, and partial dependence for interest rate, DTI and FICO were computed on the 2014 validation loans. Coefficient signs and partial-dependence directions were checked against business expectations (higher DTI, utilization, inquiries and interest rate raise predicted default risk; higher FICO and income lower it). Exceptions among logistic regression coefficients are attributed to correlated features sharing effects. Tables are in `reports/` (lr_odds_ratios, permutation_importance, shap_mean_abs) and charts in `reports/figures/` (G1 to G5).